# 10.7 · 图像分割 / Image Segmentation

> **课程定位 / Where this fits**
> 第 7 课，**Part 10 · 计算机视觉**。
> Lesson 7, **Part 10 · Computer Vision**.
>
> 检测(10.6)给的是"框"，但很多任务要求精确到**每一个像素属于什么**——这就是**图像分割**：给图里每个像素打标签。医学影像(肿瘤区域)、自动驾驶(路面/行人)、抠图都靠它。本课讲清**语义分割 vs 实例分割**、**编码器-解码器**结构、**亲手搭一个 U-Net** 在合成数据上训练、并实现**Dice 系数/损失**。
> Detection (10.6) gives boxes, but many tasks need **per-pixel labels** — that's **segmentation**: label every pixel. Medical imaging (tumor regions), self-driving (road/pedestrians), and matting rely on it. We'll cover **semantic vs instance segmentation**, the **encoder-decoder** structure, **build a U-Net** and train it on synthetic data, and implement the **Dice coefficient/loss**.
>
> 💼 **实战/面试视角**："语义vs实例分割 / U-Net 为什么有 skip / 上采样怎么做 / Dice loss" 是分割岗常考。
> 💼 **Practical/interview angle:** "semantic vs instance / why U-Net skips / how to upsample / Dice loss" — segmentation-role questions.

> 📐 **符号约定 / Notation**
> - 掩码(mask) —— 与图同尺寸的逐像素标签图 / per-pixel label map, same size as image
> - 上采样(upsampling) —— 把小特征图放大回原分辨率 / enlarge a small feature map back to full size

> 💡 **面试相关 / Interview-relevant**
> - "语义分割 vs 实例分割 vs 全景分割"（出镜率 ★★★★）
> - "U-Net 的 skip connection 作用"（★★★★★）
> - "怎么把特征图上采样(转置卷积/插值)"（★★★★）
> - "Dice loss 为什么适合分割(类别不平衡)"（★★★★）

---

## 学习目标 / Learning Objectives
1. 区分语义/实例/全景分割。
   Distinguish semantic / instance / panoptic segmentation.
2. 理解**编码器-解码器**与**上采样**。
   Understand encoder-decoder and upsampling.
3. **亲手搭 U-Net** 并理解 skip connection 的作用。
   Build U-Net and understand its skip connections.
4. 在合成数据上训练分割并可视化预测掩码。
   Train segmentation on synthetic data and visualize predicted masks.
5. 实现 **Dice 系数/损失**，知道分割评价指标。
   Implement Dice coefficient/loss; know segmentation metrics.

## 目录 / TOC
1. [分割任务的三种粒度 ⭐](#1)
2. [编码器-解码器与上采样 ⭐](#2)
3. [U-Net：搭建并训练 ⭐](#3)
4. [Dice 损失与评价 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 分割任务的三种粒度 ⭐ / Three Granularities

- **语义分割(semantic)**：给每个像素一个**类别**标签（"这是路/天空/车"），但**不区分同类的不同个体**（两辆车都标成"车"，连成一片）。
  **Semantic:** a **class** per pixel ("road/sky/car"), but **doesn't separate instances** (two cars are both just "car").
- **实例分割(instance)**：在语义之上**区分每个个体**（车1、车2分开）。= 检测 + 每个框内的像素级掩码（如 Mask R-CNN）。
  **Instance:** beyond semantic, **separates each object** (car 1 vs car 2). = detection + per-box pixel mask (e.g. Mask R-CNN).
- **全景分割(panoptic)**：语义 + 实例的统一（背景类用语义，可数物体用实例）。
  **Panoptic:** unifies both (semantic for "stuff", instance for "things").

下面用一张合成图直观对比"分类 / 检测 / 语义分割 / 实例分割"四种输出。
Let's contrast classification / detection / semantic / instance outputs on a synthetic image.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
sns.set_theme(style="white")

# 合成一张含两个圆(同类两个体)的图 / synthetic image with two circles (two instances of one class)
H = W = 64
yy, xx = np.mgrid[0:H, 0:W]
img = np.ones((H, W)) * 0.2
sem = np.zeros((H, W)); inst = np.zeros((H, W))
for k, (cy, cx, r) in enumerate([(20, 18, 9), (40, 45, 11)], start=1):
    m = (xx-cx)**2 + (yy-cy)**2 <= r**2                   # 圆形掩码 / circular mask
    img[m] = 0.85; sem[m] = 1; inst[m] = k               # 语义都=1; 实例分别=1,2 / semantic vs instance
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("输入图\n分类: '有圆'(1个标签)"); axes[0].axis("off")
axes[1].imshow(img, cmap="gray"); axes[1].set_title("检测: 画框(2个框)"); axes[1].axis("off")
for cy,cx,r in [(20,18,9),(40,45,11)]:
    axes[1].add_patch(plt.Rectangle((cx-r,cy-r),2*r,2*r,fill=False,edgecolor="lime",lw=2))
axes[2].imshow(sem, cmap="viridis"); axes[2].set_title("语义分割\n每像素类别(两圆同色)"); axes[2].axis("off")
axes[3].imshow(inst, cmap="tab10"); axes[3].set_title("实例分割\n区分个体(两圆不同色)"); axes[3].axis("off")
plt.tight_layout(); plt.show()
print("分类: 整图1标签; 检测: 每物体1框; 语义: 每像素类别(不分个体); 实例: 每像素+区分个体")


<a id="2"></a>
## 2. 编码器-解码器与上采样 ⭐ / Encoder-Decoder & Upsampling

分割的输出必须和输入**一样大**（每像素一个标签）。但 CNN 靠池化不断**缩小**特征图来提取语义。怎么调和？——**编码器-解码器(encoder-decoder)** 结构：
Segmentation output must be the **same size** as input (a label per pixel). But CNNs **shrink** feature maps via pooling to extract semantics. The fix: an **encoder-decoder**:
- **编码器(encoder)**：像普通 CNN 一样逐步下采样，提取"这是什么"的高层语义（但丢了精确位置）。
  **Encoder:** downsample like a normal CNN, extracting high-level "what" semantics (losing precise location).
- **解码器(decoder)**：逐步**上采样**把特征图放大回原分辨率，恢复"在哪里"的空间细节，最终每像素输出一个类别。
  **Decoder:** **upsample** back to full resolution, recovering "where," outputting a class per pixel.

**怎么上采样**（面试点）：两种主流方式——
**How to upsample** (interview): two main ways —
- **转置卷积(transposed conv / 反卷积)**：可学习的上采样（`nn.ConvTranspose2d`）。
  **Transposed convolution:** learnable upsampling (`nn.ConvTranspose2d`).
- **插值 + 卷积**：最近邻/双线性插值放大，再接普通卷积（更少棋盘伪影）。
  **Interpolation + conv:** nearest/bilinear upsample then a conv (fewer checkerboard artifacts).

下面演示下采样和上采样的尺寸变化。
Below we demo the size changes of down- and up-sampling.


In [ ]:
x = torch.randn(1, 8, 64, 64)                         # 输入特征图 64×64 / input feature map
down = nn.Conv2d(8, 16, 3, stride=2, padding=1)        # 步幅2卷积下采样 / stride-2 conv downsample
up_tconv = nn.ConvTranspose2d(16, 8, 2, stride=2)      # 转置卷积上采样×2 / transposed conv upsample
d = down(x); u = up_tconv(d)
print(f"下采样: {tuple(x.shape)} --stride2 conv--> {tuple(d.shape)}  (空间减半)")
print(f"上采样: {tuple(d.shape)} --transposed conv--> {tuple(u.shape)}  (空间翻倍, 回到64×64)")

# 插值上采样(无需学习) / interpolation upsampling
up_interp = nn.functional.interpolate(d, scale_factor=2, mode="bilinear", align_corners=False)
print(f"插值上采样: {tuple(d.shape)} --bilinear--> {tuple(up_interp.shape)}")
print("\n编码器下采样抓语义 → 解码器上采样恢复分辨率 → 每像素输出类别")
print("上采样两法: 转置卷积(可学习) / 插值+卷积(少棋盘伪影)")


<a id="3"></a>
## 3. U-Net：搭建并训练 ⭐ / U-Net: Build & Train

**U-Net** 是分割界最经典的编码器-解码器（最早用于医学影像）。它的杀手锏是 **skip connection(跳跃连接)**：把**编码器每一层的特征图，直接拼接到解码器对应层**。
**U-Net** is the classic segmentation encoder-decoder (originally for medical imaging). Its killer feature is **skip connections**: **concatenate each encoder layer's feature map directly into the matching decoder layer**.

**为什么需要 skip**（面试核心，呼应 10.4 ResNet 的跳连但目的不同）：编码器下采样时丢失了**精确的边界/位置细节**；解码器光靠上采样恢复不出锐利边缘。skip 把编码器里**高分辨率的细节**直接送给解码器，让分割边界**又准又清晰**。
**Why skips** (interview core; like 10.4's skips but for a different purpose): downsampling loses **precise boundary/location detail**; the decoder can't recover sharp edges from upsampling alone. Skips feed the encoder's **high-resolution detail** straight to the decoder, making boundaries **accurate and sharp**.

下面搭一个小 U-Net，在**合成的圆形分割任务**上训练：随机生成带圆的图，让网络逐像素预测"哪里是圆(前景)"。
We build a small U-Net and train it on a **synthetic circle-segmentation task**: random images with circles; the net predicts per-pixel "where is the circle (foreground)."


In [ ]:
def make_sample(H=64, W=64, rng=None):
    """随机生成一张含若干圆的图 + 对应前景掩码 / random image with circles + foreground mask."""
    rng = rng or np.random
    yy, xx = np.mgrid[0:H, 0:W]
    img = np.ones((H, W)) * rng.uniform(0.1, 0.3) + rng.normal(0, 0.05, (H, W))  # 带噪背景 / noisy background
    mask = np.zeros((H, W))
    for _ in range(rng.randint(1, 4)):                    # 1~3 个圆 / 1-3 circles
        cy, cx, r = rng.randint(12, H-12), rng.randint(12, W-12), rng.randint(6, 12)
        m = (xx-cx)**2 + (yy-cy)**2 <= r**2
        img[m] = rng.uniform(0.7, 0.95); mask[m] = 1     # 前景像素=1 / foreground=1
    return np.clip(img, 0, 1).astype(np.float32), mask.astype(np.float32)

rng = np.random.RandomState(0)
N = 300
X = np.zeros((N,1,64,64), np.float32); Y = np.zeros((N,1,64,64), np.float32)
for i in range(N):
    im, mk = make_sample(rng=rng); X[i,0]=im; Y[i,0]=mk
Xt, Yt = torch.tensor(X), torch.tensor(Y)
Xtr, Ytr, Xte, Yte = Xt[:240], Yt[:240], Xt[240:], Yt[240:]

class UNet(nn.Module):                                    # 迷你 U-Net / mini U-Net
    def __init__(s, c=16):
        super().__init__()
        cb = lambda i,o: nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.ReLU(), nn.Conv2d(o,o,3,padding=1), nn.ReLU())
        s.e1 = cb(1, c); s.e2 = cb(c, c*2)                # 编码器两层 / encoder
        s.pool = nn.MaxPool2d(2)
        s.bott = cb(c*2, c*4)                             # 瓶颈 / bottleneck
        s.up2 = nn.ConvTranspose2d(c*4, c*2, 2, stride=2); s.d2 = cb(c*4, c*2)   # 解码器(拼接后通道翻倍) / decoder
        s.up1 = nn.ConvTranspose2d(c*2, c, 2, stride=2);   s.d1 = cb(c*2, c)
        s.out = nn.Conv2d(c, 1, 1)                        # 1×1 输出每像素1个logit / per-pixel logit
    def forward(s, x):
        e1 = s.e1(x)                                      # 64×64, 细节丰富 / high-res detail
        e2 = s.e2(s.pool(e1))                             # 32×32
        b  = s.bott(s.pool(e2))                           # 16×16, 语义 / semantics
        d2 = s.d2(torch.cat([s.up2(b), e2], 1))          # 上采样后拼接 e2(skip) / upsample + skip-concat e2
        d1 = s.d1(torch.cat([s.up1(d2), e1], 1))         # 上采样后拼接 e1(skip) / upsample + skip-concat e1
        return s.out(d1)                                  # 64×64×1 / full-res output

torch.manual_seed(0); net = UNet(); opt = torch.optim.Adam(net.parameters(), 1e-3)
bce = nn.BCEWithLogitsLoss()
for ep in range(40):
    net.train(); opt.zero_grad(); loss = bce(net(Xtr), Ytr); loss.backward(); opt.step()
net.eval()
with torch.no_grad(): pred = torch.sigmoid(net(Xte))      # 预测前景概率 / predicted foreground prob
print(f"U-Net 训练完成, 最终训练损失 = {loss.item():.3f}")
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for j in range(5):
    axes[0,j].imshow(Xte[j,0], cmap="gray"); axes[0,j].axis("off")
    axes[1,j].imshow(Yte[j,0], cmap="viridis"); axes[1,j].axis("off")
    axes[2,j].imshow(pred[j,0]>0.5, cmap="viridis"); axes[2,j].axis("off")
for i,t in enumerate(["输入图","真实掩码 GT","U-Net 预测"]): axes[i,0].set_ylabel(t, fontsize=10, rotation=90); axes[i,0].axis("on"); axes[i,0].set_xticks([]); axes[i,0].set_yticks([])
fig.suptitle("U-Net 分割: 逐像素预测前景(圆); skip 让边界又准又清晰"); plt.tight_layout(); plt.show()
print("U-Net = 编码器(抓语义)+解码器(恢复分辨率)+skip(送回高分辨率细节→锐利边界)")


<a id="4"></a>
## 4. Dice 损失与评价 + 小结 ⭐ / Dice Loss & Metrics

分割常面临**类别极不平衡**：前景(如肿瘤)可能只占图的 1%，背景占 99%。这时用普通像素级交叉熵，模型"全预测背景"就能得到 99% 像素准确率——**完全没用**。
Segmentation often has **severe class imbalance**: foreground (e.g. tumor) may be 1% of pixels, background 99%. Plain per-pixel cross-entropy lets "predict all background" score 99% pixel accuracy — **useless**.

**Dice 系数** 直接衡量预测掩码与真实掩码的**重叠**（和分割的 IoU 类似）：$\text{Dice} = \dfrac{2|P \cap G|}{|P| + |G|}$，0~1，越大越好。**Dice loss = 1 − Dice**，它**只关注前景重叠**，天然抗不平衡，是分割的常用损失/指标。
The **Dice coefficient** directly measures **overlap** of predicted and true masks (like IoU for segmentation): $\text{Dice} = \dfrac{2|P \cap G|}{|P| + |G|}$, 0–1, higher is better. **Dice loss = 1 − Dice** focuses **only on foreground overlap**, naturally robust to imbalance — a common segmentation loss/metric.


In [ ]:
def dice_coef(pred, gt, eps=1e-6):
    """Dice 系数: 2|交集| / (|P|+|G|) / Dice coefficient."""
    pred = (pred > 0.5).float()
    inter = (pred * gt).sum()                             # 交集像素数 / intersection
    return ((2*inter + eps) / (pred.sum() + gt.sum() + eps)).item()

def iou_seg(pred, gt, eps=1e-6):
    pred = (pred > 0.5).float()
    inter = (pred * gt).sum(); union = pred.sum() + gt.sum() - inter
    return ((inter + eps) / (union + eps)).item()

d = dice_coef(pred, Yte); i = iou_seg(pred, Yte)
pixel_acc = ((pred>0.5).float() == Yte).float().mean().item()
fg_ratio = Yte.mean().item()
print(f"测试集前景占比 = {fg_ratio:.1%} (背景占多数 → 类别不平衡)")
print(f"像素准确率 = {pixel_acc:.3f}  (受背景主导, 看着高但不可靠)")
print(f"Dice 系数  = {d:.3f}   ← 只看前景重叠, 分割更该看这个")
print(f"IoU(分割)  = {i:.3f}")
print("\nDice loss = 1 - Dice; 只关注前景重叠, 抗类别不平衡 → 分割常用损失/指标")


```
分割粒度: 语义(每像素类别,不分个体) / 实例(区分个体=检测+掩码) / 全景(两者统一)
结构: 编码器(下采样抓语义) + 解码器(上采样恢复分辨率, 每像素输出类别)
上采样: 转置卷积(可学习) / 插值+卷积(少棋盘伪影)
U-Net: 编码器-解码器 + skip(把编码器高分辨率细节拼到解码器→边界锐利准确)
Dice: 2|P∩G|/(|P|+|G|), 只看前景重叠抗不平衡; Dice loss=1-Dice; 别只看像素准确率
代表模型: FCN / U-Net(语义) / Mask R-CNN(实例) / DeepLab(空洞卷积)
```

### 💡 面试速查 / Interview cheat-sheet
1. **语义 vs 实例**: 是否区分同类个体(实例=检测+掩码)。
   Semantic vs instance: whether instances are separated (instance = detection + mask).
2. **编码器-解码器**: 下采样抓语义→上采样回原分辨率逐像素分类。
   Encoder-decoder: downsample for semantics → upsample to per-pixel labels.
3. **U-Net skip**: 送回高分辨率细节, 让边界锐利准确。
   U-Net skips: pass high-res detail for sharp, accurate boundaries.
4. **上采样**: 转置卷积或插值+卷积。
   Upsampling: transposed conv or interpolation+conv.
5. **Dice loss**: 看前景重叠, 抗类别不平衡(别只看像素准确率)。
   Dice loss: foreground overlap, robust to imbalance (don't trust pixel accuracy alone).

### 下一节 / Next
**10.8 关键点检测 / 姿态估计**——不只框和掩码, 还要定位物体上的**关键点**(人的关节、脸的特征点)。核心技巧是**热图回归**。我们会用合成数据演示。
**10.8 Pose Estimation** — beyond boxes and masks, locate **keypoints** (human joints, facial landmarks). The core trick is **heatmap regression**, which we demo on synthetic data.
